# Deep Agents 하이브리드 에이전트 — RAG + MCP + Skills

`create_deep_agent`로 세 가지 핵심 기술을 하나의 에이전트에 통합한다.

| 시나리오 | 기대 동작 | 사용 기술 |
|----------|----------|----------|
| 회사 사내 규정 질문 | RAG 벡터 검색 → 문서 기반 답변 | `retrieve` 도구 |
| 텍스트 분석 요청 | MCP 서버 호출 → 글자/단어 수 계산 | `count_chars`, `count_words` 도구 |
| 보고서 작성 요청 | SKILL.md 양식 로드 → 양식에 맞는 출력 | `skills=` 파라미터 |


In [1]:
# 환경 설정
import os
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

model = ChatOpenAI(model="gpt-5.4-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print("환경 준비 완료.")


환경 준비 완료.


---
## Step 1. RAG — 회사 사내 규정 지식 베이스

회사 HR 규정 문서를 **Chroma 벡터 스토어**에 인덱싱하고, `retrieve` 도구로 검색한다.


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.tools import tool

# 메인: Chroma 사용
from langchain_chroma import Chroma

# (대안 1) FAISS 사용 시:
# from langchain_community.vectorstores import FAISS

# (대안 2) 내장 인메모리 사용 시 (영속성 없음, 학습용):
# from langchain_core.vectorstores import InMemoryVectorStore

# 주간 보고서 작성에 필요한 사내 자료
company_docs = [
    Document(
        page_content="[보고서 작성 가이드] 주간 보고서는 매주 금요일 17시까지 팀 채널에 공유한다. "
        "보고서에는 금주 실적, 주요 이슈, 차주 계획, 건의사항 네 섹션이 반드시 포함되어야 한다. "
        "각 항목은 불릿 포인트로 간결하게 작성하고, 가능한 정량 지표(완료율, 처리 건수, 일정)를 함께 적는다. "
        "보고 기간은 월요일 ~ 금요일 기준으로 표기한다.",
        metadata={"source": "GUIDE-001 주간보고서 작성 가이드"},
    ),
    Document(
        page_content="[금주 프로젝트 현황] RAG 파이프라인 구축 작업이 80% 완료되었으며, "
        "Chroma 벡터 스토어 인덱싱과 retrieve 도구 통합을 마쳤다. "
        "MCP 서버 연동 테스트는 stdio 방식으로 정상 동작을 확인했고, 글자/단어 수 카운트 도구가 추가되었다. "
        "프롬프트 엔지니어링은 v1(단순)과 v2(규칙 명시) 두 버전을 비교 실험 중이다. "
        "전체 진척도 약 70%, 담당자: 김순주.",
        metadata={"source": "PROJ-024 금주 프로젝트 현황"},
    ),
    Document(
        page_content="[금주 주요 이슈] langchain-chroma 모듈 미설치로 초기 인덱싱 실패가 있었으나 "
        "pyproject.toml에 langchain-chroma, chromadb 의존성을 추가한 뒤 uv sync로 해결했다. "
        "또한 SKILL.md가 LLM 판단으로 발동되지 않는 케이스가 있어, 시스템 프롬프트에 "
        "'보고서 작성 시 weekly-report 스킬을 반드시 따른다' 규칙을 명시하는 v2 버전을 도입했다.",
        metadata={"source": "ISSUE-2024W12 금주 이슈"},
    ),
    Document(
        page_content="[차주 계획] 다음 주에는 하이브리드 에이전트의 프로덕션 배포를 진행한다. "
        "월요일에 스테이징 환경에 배포 후 사내 베타 테스터 5명과 함께 회귀 테스트를 수행하고, "
        "수요일까지 발견된 버그를 수정한다. 목요일에 운영 환경 배포, 금요일에 릴리스 노트 공유 예정. "
        "추가로 SKILL.md 2종(회의록 작성, 코드 리뷰)을 신규 작성한다.",
        metadata={"source": "PLAN-2024W13 차주 계획"},
    ),
    Document(
        page_content="[건의사항/리소스 요청] 운영 배포를 위한 GPU 인스턴스 1대 추가 요청이 필요하다. "
        "현재 임베딩 모델 호출이 OpenAI API에 의존하고 있어 월 비용 모니터링이 필요하며, "
        "자체 임베딩 서버 도입 검토를 인프라팀에 공식 요청할 예정. "
        "또한 SKILL.md 표준 양식 정립을 위한 1시간 협업 미팅을 다음 주 화요일에 제안한다.",
        metadata={"source": "REQ-018 건의사항"},
    ),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
splits = splitter.split_documents(company_docs)

# 메인: Chroma 벡터 스토어
vector_store = Chroma.from_documents(splits, embedding=embeddings, collection_name="weekly_report_kb")

# (대안 1) FAISS:
# vector_store = FAISS.from_documents(splits, embeddings)

# (대안 2) 내장 인메모리:
# vector_store = InMemoryVectorStore.from_documents(splits, embeddings)

@tool(response_format="content_and_artifact")
def retrieve(query: str):
    """주간 보고서 작성에 필요한 사내 자료(가이드, 프로젝트 현황, 이슈, 계획, 건의사항)를 검색한다."""
    results = vector_store.similarity_search(query, k=3)
    text = "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in results)
    return text, results

print(f"주간 보고서 KB 벡터 스토어(Chroma) 구축: {len(splits)}개 청크")


주간 보고서 KB 벡터 스토어(Chroma) 구축: 5개 청크


### 터미널 — Math 서버

cd 06-hybrid  

uv run python math_server.py

→ `http://localhost:8001/mcp` 에서  시작

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# ──────────────────────────────────────────────────────────────────
# 클라이언트에 텍스트 유틸 서버 등록
# ──────────────────────────────────────────────────────────────────
mcp_client = MultiServerMCPClient(
    {
        "text": {
            "transport": "streamable_http",
            "url": "http://localhost:8001/mcp",
            # 상용 MCP 서버 인증이 필요하면:
            # "headers": {"Authorization": "Bearer YOUR_TOKEN"},
        },
    }
)

# 모든 서버의 도구를 한 번에 가져오기 (await 필수)  
mcp_tools = await mcp_client.get_tools()  
print(f"MCP 도구: {[t.name for t in mcp_tools]}")
  

MCP 도구: ['count_chars', 'count_words']


---
## Step 3. Skills — SKILL.md로 보고서 양식 정의

`SKILL.md` 파일을 디스크에 생성하고, `create_deep_agent`의 `skills=` 파라미터로 로드한다.


In [4]:
import tempfile
from pathlib import Path

workspace = tempfile.mkdtemp()
skill_dir = Path(workspace) / "skills" / "weekly-report"
skill_dir.mkdir(parents=True)

skill_md = skill_dir / "SKILL.md"
skill_md.write_text("""\
---
name: weekly-report
description: 주간 업무 보고서를 회사 표준 양식으로 작성한다. 보고서, 리포트, 주간 보고 요청 시 사용한다.
---

# 주간 보고서 작성 스킬

## 사용 시기
- 사용자가 주간 보고서, 업무 보고, 주간 리포트 작성을 요청할 때

## 보고서 양식 (반드시 아래 형식을 따를 것)

### [주간 업무 보고서]
- **보고 기간**: YYYY.MM.DD ~ YYYY.MM.DD
- **작성자**: (사용자 이름 또는 '작성자')

#### 1. 금주 실적
- 완료된 업무를 불릿 포인트로 정리

#### 2. 주요 이슈
- 발생한 문제점과 해결 방안

#### 3. 차주 계획
- 다음 주 예정 업무

#### 4. 건의사항
- 지원 요청 또는 건의

## 규칙
- 각 섹션은 반드시 포함할 것
- 불릿 포인트로 간결하게 작성
- 구체적 날짜와 수치를 포함
""", encoding="utf-8")

print(f"SKILL.md 생성: {skill_md}")
print(f"워크스페이스: {workspace}")


SKILL.md 생성: C:\Users\Soonju\AppData\Local\Temp\tmpd77taljg\skills\weekly-report\SKILL.md
워크스페이스: C:\Users\Soonju\AppData\Local\Temp\tmpd77taljg


---
## Step 4. Deep Agent 생성 — RAG + MCP + Skills 통합

두 가지 버전을 만든다:
- **v1 (단순 프롬프트)**: 시스템 프롬프트에 도구 사용 규칙 없음 → LLM이 알아서 판단
- **v2 (명시 규칙)**: 시스템 프롬프트에 "어떤 도구를 언제 쓸지" 명시

뒤에서 같은 질문에 두 버전의 동작을 비교한다.


In [5]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

all_tools = [retrieve] + mcp_tools

# ── v1: 단순 프롬프트 ───────────────────────────────────
hybrid_agent = create_deep_agent(
    model=model,
    tools=all_tools,
    system_prompt="당신은 회사 AI 어시스턴트다. 한국어로 답변한다.",
    skills=["/skills/"],
    backend=FilesystemBackend(root_dir=workspace, virtual_mode=True),
)

print("v1 (단순) 에이전트 생성 완료")
print(f"  도구: {[t.name for t in all_tools]}")
print(f"  스킬: /skills/weekly-report/SKILL.md")


v1 (단순) 에이전트 생성 완료
  도구: ['retrieve', 'count_chars', 'count_words']
  스킬: /skills/weekly-report/SKILL.md


In [6]:
# ── v2: 도구 사용 규칙 명시 ─────────────────────────────
hybrid_agent_v2 = create_deep_agent(
    model=model,
    tools=all_tools,
    system_prompt="""당신은 회사 AI 어시스턴트다. 한국어로 답변한다.

## 도구 사용 규칙
- 회사 규정/정책 질문 → retrieve 도구로 사내 규정을 검색한다
- 글자 수/단어 수 계산 → count_chars, count_words 도구를 사용한다 (직접 세지 않는다)
- 보고서 작성 → weekly-report 스킬의 양식을 반드시 따른다

## 머릿말
-"v2 시스템프롬프트 적용 결과입니다."를 제일 앞에 항상 출력한다.

## 톤
- 7살 어린아이에게 알려주듯 쉬운 문장으로 풀어 말한다.
""",

    skills=["/skills/"],
    backend=FilesystemBackend(root_dir=workspace, virtual_mode=True),
)

print("v2 (규칙 명시) 에이전트 생성 완료")


v2 (규칙 명시) 에이전트 생성 완료


---
## Step 5. 헬퍼 — 사용된 스킬/도구 추출


In [7]:
import re

def extract_used_skills(messages) -> list[str]:
    """tool_calls를 훑어 SKILL.md를 읽은 흔적이 있는지 본다."""
    used = []
    for m in messages:
        for call in getattr(m, "tool_calls", []) or []:
            for v in (call.get("args") or {}).values():
                if isinstance(v, str):
                    m_ = re.search(r"skills[/\\]([^/\\]+)[/\\]SKILL\.md", v)
                    if m_:
                        used.append(m_.group(1))
    seen, ordered = set(), []
    for s in used:
        if s not in seen:
            seen.add(s); ordered.append(s)
    return ordered

def extract_used_tools(messages) -> list[str]:
    names = []
    for m in messages:
        for call in getattr(m, "tool_calls", []) or []:
            n = call.get("name")
            if n:
                names.append(n)
    return names

def show(result, title):
    msgs = result["messages"]
    print("=" * 60)
    print(f"[{title}]")
    print("=" * 60)
    print("호출된 도구 :", extract_used_tools(msgs) or "(없음)")
    print("발동된 스킬:", extract_used_skills(msgs) or "(없음)")
    print("-" * 60)
    print(msgs[-1].content)
    print()


---
## Step 6. 테스트 — v1 (단순 프롬프트)


In [8]:
# 테스트 1: RAG — 사내 자료 조회
result = await hybrid_agent.ainvoke(
    {"messages": [{"role": "user", "content":
        "이번 주 프로젝트 진척도가 어떻게 돼? 사내 자료 보고 알려줘."}]}
)
show(result, "v1 / RAG")


[v1 / RAG]
호출된 도구 : ['write_todos', 'retrieve', 'write_todos', 'write_todos']
발동된 스킬: (없음)
------------------------------------------------------------
이번 주 프로젝트 진척도는 **전체 약 70%**입니다.

- **금주 실적**
  - RAG 파이프라인 구축 작업 **80% 완료**
  - **Chroma 벡터 스토어 인덱싱** 및 **retrieve 도구 통합 완료**
  - **MCP 서버 연동 테스트**는 stdio 방식으로 정상 동작 확인
  - **글자/단어 수 카운트 도구 추가**
  - 프롬프트 엔지니어링 **v1/v2 비교 실험 진행 중**

- **주요 이슈**
  - 프롬프트 엔지니어링 버전별 결과 비교가 아직 진행 중
  - 전체 진척도는 높지만, 최종 품질 검증은 추가 필요

- **차주 계획**
  - 하이브리드 에이전트 **프로덕션 배포**
  - 월요일: 스테이징 배포 후 **베타 테스터 5명과 회귀 테스트**
  - 수요일까지 버그 수정
  - 목요일 운영 배포
  - 금요일 릴리스 노트 공유

- **건의사항**
  - 회귀 테스트 결과와 버그 수정 우선순위를 빠르게 정리하면 배포 리스크를 줄일 수 있음

원하시면 이 내용을 **주간보고서 형식**으로 바로 정리해드릴게요.



In [9]:
# 테스트 2: MCP — 텍스트 분석
result = await hybrid_agent.ainvoke(
    {"messages": [{"role": "user", "content": "다음 문장의 글자 수와 단어 수를 알려줘: '오늘은 회사에서 보고서를 작성하고 있습니다.'"}]}
)
show(result, "v1 / MCP")


[v1 / MCP]
호출된 도구 : ['count_chars', 'count_words']
발동된 스킬: (없음)
------------------------------------------------------------
글자 수: 24자  
단어 수: 5개



In [10]:
# 테스트 3: Skill + RAG + MCP 통합 — 보고서 작성
result = await hybrid_agent.ainvoke(
    {"messages": [{"role": "user", "content":
        "사내 자료를 참고해서 이번 주 주간 보고서를 작성해줘. "
        "금주 실적, 주요 이슈, 차주 계획, 건의사항을 모두 포함하고, "
        "마지막에 보고서 본문의 글자 수도 알려줘."}]}
)
show(result, "v1 / Skill")


[v1 / Skill]
호출된 도구 : ['write_todos', 'read_file', 'retrieve', 'write_todos', 'count_chars', 'write_todos']
발동된 스킬: ['weekly-report']
------------------------------------------------------------
[주간 업무 보고서]
- 보고 기간: 2024.03.25 ~ 2024.03.29
- 작성자: 작성자

#### 1. 금주 실적
- RAG 파이프라인 구축 80% 완료, Chroma 벡터 스토어 인덱싱 및 retrieve 도구 통합 완료
- MCP 서버 연동 테스트를 stdio 방식으로 정상 확인
- 글자 수/단어 수 카운트 도구를 추가하여 보고서 보조 기능 확장
- 프롬프트 엔지니어링 v1/v2 비교 실험을 진행하며 개선 방향 검토

#### 2. 주요 이슈
- 프롬프트 엔지니어링 v1/v2 비교 실험 결과를 바탕으로 최종안을 확정해야 함
- RAG 파이프라인의 남은 20% 구간에서 성능 검증과 안정화가 필요함
- 신규 기능 추가에 따른 문서화 및 검증 범위가 확대됨

#### 3. 차주 계획
- 월요일에 하이브리드 에이전트 스테이징 배포 후 사내 베타 테스터 5명 대상 회귀 테스트 수행
- 수요일까지 발견 버그 수정 후 목요일 운영 환경 배포 진행
- 금요일 릴리스 노트 공유 예정
- 신규 문서 2종(Skill 작성 관련) 초안 작성 착수

#### 4. 건의사항
- 배포 전 회귀 테스트를 위한 베타 테스터 일정 조율 지원 요청
- 문서화 및 검증 업무 병행을 위해 단기 리뷰 리소스 배정 건의

보고서 본문 글자 수: 621자



---
## Step 7. 테스트 — v2 (도구 사용 규칙 명시)

같은 세 질문을 v2 에이전트에 넣어 도구/스킬 선택 차이를 본다.


In [11]:
# 테스트 1: RAG — 사내 자료 조회
result = await hybrid_agent_v2.ainvoke(
    {"messages": [{"role": "user", "content":
        "이번 주 프로젝트 진척도가 어떻게 돼? 사내 자료 보고 알려줘."}]}
)
show(result, "v2 / RAG")


[v2 / RAG]
호출된 도구 : ['retrieve']
발동된 스킬: (없음)
------------------------------------------------------------
v2 시스템프롬프트 적용 결과입니다.

사내 자료를 보면, 이번 주 프로젝트 진척도는 **약 70%**예요.

- **RAG 파이프라인 구축**: **80% 완료**
- **완료한 일**
  - Chroma 벡터 스토어 인덱싱 완료
  - `retrieve` 도구 통합 완료
  - MCP 서버 연동 테스트에서 **stdio 방식 정상 동작 확인**
  - 글자/단어 수 카운트 도구 추가
- **진행 중인 일**
  - 프롬프트 엔지니어링 **v1(단순) vs v2(규칙 명시)** 비교 실험 중

한마디로 말하면, **핵심 기능은 꽤 많이 됐고, 지금은 프롬프트를 더 잘 다듬는 중**이에요.



In [12]:
### Step6 단순프롬프트 결과 비교

# v2 (규칙 명시) 에이전트 생성 완료
# ============================================================
# [v1 / RAG]
# ============================================================
# 호출된 도구 : ['write_todos', 'retrieve', 'write_todos']
# 발동된 스킬: (없음)
# ------------------------------------------------------------
# 이번 주 프로젝트 진척도는 **전체 약 70%**입니다.

# - **금주 실적**
#   - RAG 파이프라인 구축 작업 **80% 완료**
#   - **Chroma 벡터 스토어 인덱싱 완료**
#   - **retrieve 도구 통합 완료**
#   - **MCP 서버 연동 테스트(stdio) 정상 확인**
#   - **글자/단어 수 카운트 도구 추가 완료**

# - **진행 중**
#   - 프롬프트 엔지니어링 **v1(단순) vs v2(규칙 명시)** 비교 실험 진행 중

# - **주요 이슈**
#   - 현재 자료상 큰 장애 이슈는 없고, 비교 실험 결과를 바탕으로 프롬프트 방향을 확정해야 함

# - **차주 계획**
#   - 하이브리드 에이전트 **프로덕션 배포**
#   - 월요일: 스테이징 배포 및 **사내 베타 테스터 5명 회귀 테스트**
#   - 수요일까지: 발견 버그 수정
# ...
#   - 금요일: 릴리스 노트 공유

# 원하시면 이 내용을 **주간 보고서 형식**으로 바로 정리해드릴게요.

In [13]:
# 테스트 2: MCP
result = await hybrid_agent_v2.ainvoke(
    {"messages": [{"role": "user", "content": "다음 문장의 글자 수와 단어 수를 알려줘: '오늘은 회사에서 보고서를 작성하고 있습니다.'"}]}
)
show(result, "v2 / MCP")


[v2 / MCP]
호출된 도구 : ['count_chars', 'count_words']
발동된 스킬: (없음)
------------------------------------------------------------
v2 시스템프롬프트 적용 결과입니다.
글자 수는 24자예요.  
단어 수는 5개예요.



In [14]:
# ============================================================
# [v1 / MCP]
# ============================================================
# 호출된 도구 : ['count_chars', 'count_words']
# 발동된 스킬: (없음)
# ------------------------------------------------------------
# 글자 수: 24자  
# 단어 수: 5개

In [15]:
# 테스트 3: Skill + RAG + MCP 통합 — 보고서 작성
result = await hybrid_agent_v2.ainvoke(
    {"messages": [{"role": "user", "content":
        "사내 자료를 참고해서 이번 주 주간 보고서를 작성해줘. "
        "금주 실적, 주요 이슈, 차주 계획, 건의사항을 모두 포함하고, "
        "마지막에 보고서 본문의 글자 수도 알려줘."}]}
)
show(result, "v2 / Skill")


[v2 / Skill]
호출된 도구 : ['write_todos', 'task', 'count_chars', 'write_todos']
발동된 스킬: ['weekly-report']
------------------------------------------------------------
v2 시스템프롬프트 적용 결과입니다.

[주간 업무 보고서]

- 보고 기간: 2026-04-06 ~ 2026-04-10
- 작성자: AI 어시스턴트

1. 금주 실적
- 주간 보고서 작성 가이드와 표준 양식을 확인하고 보고 구조를 정리함.
- `weekly-report` 스킬 적용 기준과 필수 항목을 확인함.
- 사내 자료를 바탕으로 금주 실적, 주요 이슈, 차주 계획, 건의사항을 정리할 수 있는 상태로 준비함.

2. 주요 이슈
- `langchain-chroma` 미설치로 초기 인덱싱 실패가 발생했으나, `pyproject.toml`에 `langchain-chroma`, `chromadb` 의존성을 추가하고 `uv sync`로 해결함.
- `weekly-report` 스킬이 LLM 판단으로 누락될 수 있는 문제를 줄이기 위해 시스템 프롬프트 규칙을 강화한 v2를 적용함.

3. 차주 계획
- 하이브리드 에이전트 프로덕션 배포를 진행함.
- 월요일에 스테이징 배포 후 사내 베타 테스터 5명을 대상으로 회귀 테스트를 진행함.
- 수요일까지 발견된 버그를 수정함.
- 목요일에 운영 배포를 진행함.
- 금요일에 릴리스 노트를 공유함.
- SKILL.md 2종(회의록 작성, 코드 리뷰) 신규 작성 작업을 진행함.

4. 건의사항
- 배포 전 의존성 점검과 회귀 테스트를 더 꼼꼼히 보면 좋겠음.
- 스킬 적용 누락을 줄이기 위해 작성 가이드와 시스템 규칙을 계속 점검하면 좋겠음.

보고서 본문의 글자 수는 701자입니다.



In [16]:
# ============================================================
# [v1 / Skill]
# ============================================================
# 호출된 도구 : ['write_todos', 'retrieve', 'retrieve', 'count_chars', 'write_todos', 'count_chars', 'write_todos']
# 발동된 스킬: (없음)
# ------------------------------------------------------------
# [보고 기간] 월요일 ~ 금요일

# 금주 실적
# - RAG 파이프라인 구축 작업 80% 완료, 전체 진척도 약 70%.
# - Chroma 벡터 스토어 인덱싱 및 retrieve 도구 통합 완료.
# - MCP 서버 연동 테스트를 stdio 방식으로 정상 확인.
# - 글자 수/단어 수 카운트 도구 추가.
# - 프롬프트 엔지니어링 v1(단순)과 v2(규칙 명시) 비교 실험 진행.

# 주요 이슈
# - langchain-chroma 모듈 미설치로 초기 인덱싱 실패가 있었으나, 의존성 추가 및 uv sync로 해결.
# - SKILL.md가 LLM 판단으로 발동되지 않는 케이스가 있어, 시스템 프롬프트에 보고서 작성 시 weekly-report 스킬 준수 규칙을 명시하는 v2를 도입.
# - 하이브리드 에이전트 프로덕션 배포를 앞두고 있어, 스테이징/회귀 테스트 결과에 따라 일정 변동 가능.

# 차주 계획
# - 월요일: 스테이징 환경 배포 및 사내 베타 테스터 5명과 회귀 테스트 수행.
# - 수요일까지 발견된 버그 수정.
# - 목요일: 운영 환경 배포.
# - 금요일: 릴리스 노트 공유.
# ...
# - 운영 배포 전 체크리스트를 배포/롤백 기준까지 포함한 형태로 사전 공유하면 리스크를 줄일 수 있음.

# 보고서 본문 글자 수: 687자


---
## 핵심 정리

| 구성 요소 | Deep Agents 파라미터 | 역할 |
|-----------|---------------------|------|
| **RAG** | `tools=[retrieve]` | 사내 규정 Chroma 벡터 검색 → 문서 기반 답변 |
| **MCP** | `tools=[...mcp_tools]` | 외부 MCP 서버 도구 호출 (글자/단어 수 계산) |
| **Skills** | `skills=["/skills/"]` | SKILL.md 양식 로드 → 구조화된 출력 |
| **Backend** | `backend=FilesystemBackend(...)` | 스킬 파일 접근용 파일시스템 |

### v1 vs v2 비교 포인트
- **v1**: 시스템 프롬프트 단순 → LLM이 도구를 알아서 판단. 가끔 직접 답해버릴 수 있음.
- **v2**: 규칙 명시 → 도구/스킬 호출이 더 안정적. 회사 운영용으로 권장.
